# Unsloth Studio on Kaggle

Run Unsloth Studio on Kaggle's free GPUs and open it from your browser through a Cloudflare tunnel. Search any GGUF model in the UI, load it and chat.

**Before you start:** set the accelerator to **GPU T4 x2** and turn **Internet** on, then run the cells in order.

## 1. Install (once per session)

In [1]:
!curl -fsSL https://unsloth.ai/install.sh -o /tmp/u.sh
!yes n | sh /tmp/u.sh 2>&1 | tail -n 5

                 source '/root/.unsloth/studio/unsloth_studio/bin/activate'
                 unsloth studio -p 8888
                 (add -H 0.0.0.0 for LAN / cloud access; exposes the raw port only, not a public URL)
                 (add -H 0.0.0.0 --cloudflare for a public Cloudflare HTTPS link, or --secure to keep the raw port private; anyone with the API key can run code)
yes: 
standard output: Broken pipe


## 2. Start Studio
Uses port 8899 because Kaggle's Jupyter already uses 8888. You want `200` at the end.

In [2]:
import os, subprocess, time
os.environ["PATH"] += ":" + os.path.expanduser("~/.local/bin")
os.environ["HF_HOME"] = "/tmp/hf"   # keep model downloads out of /kaggle/working

studio = subprocess.Popen(
    "unsloth studio -H 127.0.0.1 -p 8899 > /tmp/studio.log 2>&1",
    shell=True,
)
time.sleep(40)
!curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8899

200


## 3. Get the public link
The link changes every time you restart the tunnel, and anyone with it can reach your login page.

In [3]:
import re, subprocess
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf && chmod +x /tmp/cf
t = subprocess.Popen(["/tmp/cf", "tunnel", "--url", "http://127.0.0.1:8899"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in t.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print(m.group(0))
        break

https://herbal-sociology-rotation-fitting.trycloudflare.com


## 4. First-login password
Open the link, paste this value into **Current password**, then set your own new password.

In [4]:
!cat ~/.unsloth/studio/auth/.bootstrap_password

DemeaningGongArrivalEleven


## Notes
- Keep this session running, because closing it stops Studio and the link.
- If `unsloth` is not found, the install did not finish. Rerun cell 1.
- If a model runs out of memory, pick a smaller quant or lower the context length in Studio.